<a href="https://colab.research.google.com/github/Laiba-saeed92/Deep-Learning-and-NLP-Projects/blob/main/Text_emotion_detection_usingLSTM_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IMPORTING  LIBRARIES**

In [ ]:
# Importing the important machine learning, deep learning libraraies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score ,confusion_matrix, f1_score, recall_score, precision_score
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
import tensorflow as tf
import keras
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences,to_categorical
from keras.models import Sequential
from keras.layers import Dense,Embedding,SimpleRNN,LSTM,GRU,Bidirectional

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # To ignore warnings

In [ ]:
from google.colab import files
files.upload()  # upload your kaggle.json file here


In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d parulpandey/emotion-dataset


In [ ]:
!unzip emotion-dataset.zip


In [ ]:
import pandas as pd
train= pd.read_csv('training.csv')
test= pd.read_csv('test.csv')
val= pd.read_csv('validation.csv')

# **Exploratory Data Anakysis**

In [ ]:
#Printing 1-5 rows of test, train, validation data
print("Train", train.head())
print("Test", test.head())
print("Validation", val.head())

In [ ]:
#Printing shape of train, test, validation data
print("Train shape", train.shape)
print("Test shape", test.shape)
print("Validation shape", val.shape)

In [ ]:
#We will perform EDA on the train data
train.info()

In [ ]:
train.describe()

In [ ]:
train['label'].value_counts()


In [ ]:
train['label'].value_counts().plot(kind='bar')

In [ ]:
#Feature engineering and EDA
'''df.isnull().sum() → EDA
df.dropna() or df.fillna() → Feature engineering
df.duplicated().sum() → EDA
df.drop_duplicates() → Feature engineering'''
print("The null value rows in the training dataset is:", train.isnull().sum())
print("The duplicate rows in the training dataset is:", train.duplicated().sum())

In [ ]:
train.drop_duplicates(keep='first', inplace=True) ## Dropping the duplicated values and keeping the first value in the dataset
print("The duplicate rows in the training dataset is:", train.duplicated().sum()) #It will show no duplicated values after fropping it
print("New shape of train dataset:", train.shape)



In [ ]:
#Text Processing
nltk.download("stopwords")
# .*? Match any character (.), zero or more times (*), but as few as possible (?) — this is called non-greedy or lazy matching,<.*?> → matches things like <p>, <a href="...">, <div class="text">, etc.
pattern= re.compile('<.*?>') #re → Python’s built-in regular expression (regex) module,re.compile() -> creates a regex pattern object that you can reuse efficiently for searching, matching, or replacing text.
punctuation= string.punctuation #a predefined constant that contains all standard punctuation characters.e.g !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
ps= PorterStemmer() ## Creating a PorterStemmer object for the stemming purpose
tokenizer= Tokenizer() # Creating a tokenizer object for converting text into numeric form



In [ ]:
def preprocess_data(text):
  text= re.sub(pattern, " ", text) #Removes unwanted patterns (like HTML tags, URLs, or special characters)

  text= text.lower() #Converts all characters to lowercase.str.maketrans('', '', punctuation) creates a translation map that deletes all punctuation.

  text = text.translate(str.maketrans('','',punctuation))   # Removing all the punctuation from the text

  text= text.split() #word tokenize the text splits the sentence into individual words (tokens).Example: "hello world" → ["hello", "world"]

  text= [ps.stem(word) for word in text if not word in stopwords.words('english')] #removes stopwords and applies stemming,stopwords.words('english') → a list of common words like “the”, “is”, “and”.
  #ps.stem(word) → reduces words to their root form: “playing”, “played”, “plays” → “play”

  return " ". join(text) #Joins the list of cleaned words back into a single string, without spaces



In [ ]:
'''train['text'] This accesses the entire “text” column of your DataFrame. for word in train['text'] Iterates over each row (text entry) in that column
Calls your preprocessing function on each row — cleaning, tokenizing, removing stopwords, stemming, etc.Each cleaned text (string) is returned.it creates one list where each element corresponds to one cleaned text row from your DataFrame.
  train['text'] =
0   I am happy today!
1   I am so angry!!!
2   What a lovely day.
 processed_train = [
   "happy today",
   "angri",
    "love day"
    ]
You can also add this processed text back to your DataFrame:e.g train['clean_text'] = train['text'].apply(preprocess_data)'''
processed_train= [preprocess_data(sentence) for sentence in train['text']]
processed_test= [preprocess_data(sentence) for sentence in test['text']]
processed_validation= [preprocess_data(sentence) for sentence in val['text']]

In [ ]:
#Now Putting all the processed text of train data into a whole text. Then fit this whole text into the tokenizer for word embedding.
#The purpose of that code is to make one large string containing all sentences, so the tokenizer sees every word in the training data at once.
#processed_train_data = ["love cat", "today be sunni", "movie great"]-->whole_text = ["love cattoday be sunnimovie great"]
whole_text=" "
for i in processed_train:
  whole_text= whole_text+i
tokenizer.fit_on_texts([whole_text])
print(len(tokenizer.word_index))


In [ ]:
#Now the train processed texts are converted into respective numeric sequences which are further padded to have equal sizes.
x_train_sequences=[]
for i in processed_train:
  x_train_sequences.append(tokenizer.text_to_sequences([i][0]))
x_train_pad= pad_sequences(x_train_sequences, maxlen=50, padding='post')
x_train= np.array(x_train_pad)
y_train= np.array(to_categorical(train['label']))


In [ ]:
#Now in this cell the validation processed texts are converted into respective numeric sequences which are further padded to have equal sizes.

x_validation_sequences = []

for i in processed_validation:
  x_validation_sequences.append(tokenizer.texts_to_sequences([i])[0])    # Each processed text is converted into sequences

x_validation_pad = pad_sequences(x_validation_sequences,maxlen = 50, padding = 'post')    # Each sequences are padded to have equal size.

x_validation = np.array(x_validation_pad)
y_validation = np.array(to_categorical(val['label']))

In [ ]:
#Now in this cell the test processed texts are converted into respective numeric sequences which are further padded to have equal sizes.

x_test_sequences = []

for i in processed_test:
  x_test_sequences.append(tokenizer.texts_to_sequences([i])[0])     # Each processed text is converted into sequences

x_test_pad = pad_sequences(x_test_sequences,maxlen = 50, padding = 'post')    # Each sequences are padded to have equal size.

x_test = np.array(x_test_pad)
y_test = np.array(test['label'])

# *Model Building And Training*

# **SIMPLE LSTM MODEL**

In [ ]:
#Setting the hyperparameter for embedding layer
vocab_size=
dims= 50
maxlen= 50
# LSTM model
lstm_model = Sequential()
lstm_model.add(Embedding(vocab_size,dims,input_length = maxlen))
lstm_model.add(LSTM(100,activation='relu'))
lstm_model.add(Dense(6,activation = 'softmax'))

lstm_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

lstm_model.summary()


In [ ]:
#Train the LSTM model
# Create EarlyStopping callback
early_stop = EarlyStopping(
    monitor='val_loss',       # What metric to monitor (can be 'val_accuracy', etc.)
    patience=3,               # How many epochs to wait for improvement before stopping
    restore_best_weights=True # Restores weights from the epoch with best monitored metric
)
lstm_model_history = lstm_model.fit( x_train, y_train, validation_data = (x_validation,y_validation), epochs = 10, batch_size = 32, callbacks=[early_stop])

# **BIDIRECTIONAL LSTM MODEL**

In [ ]:
## Bidirectional LSTM model

bidirectional_lstm_model = Sequential()

bidirectional_lstm_model.add(Embedding(vocab_size,dims ,input_length = maxlen))
bidirectional_lstm_model.add(Bidirectional(LSTM(100)))
bidirectional_lstm_model.add(Dense(6,activation = 'softmax'))

bidirectional_lstm_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
bidirectional_lstm_model.summary()

In [ ]:
# Training the Bidirectional LSTM model

bidirectional_lstm_model_history = bidirectional_lstm_model.fit( x_train, y_train, validation_data = (x_validation,y_validation), epochs = 10, batch_size = 32,  callbacks=[early_stop])

# **STACKED LSTM MODEL**

In [ ]:
# Stacked LSTM model

stack_lstm_model = Sequential()
stack_lstm_model.add(Embedding(vocab_size,dims ,input_length = maxlen))
stack_lstm_model.add(LSTM(100,return_sequences = True))
stack_lstm_model.add(LSTM(100,return_sequences = True))
stack_lstm_model.add(LSTM(50))
stack_lstm_model.add(Dense(6,activation = 'softmax'))

stack_lstm_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

stack_lstm_model.summary()

In [ ]:
# Training the Bidirectional LSTM model
stack_lstm_model_history =  stack_lstm_model.fit( x_train, y_train, validation_data = (x_validation,y_validation), epochs = 10, batch_size = 32, callbacks=[early_stop])

# **SIMPLE GRU MODEL**

In [ ]:
# Simple GRU model
gru_model = Sequential()
gru_model.add(Embedding(vocab_size,dims ,input_length =maxlen))
gru_model.add(GRU(100))
gru_model.add(Dense(6,activation = 'softmax'))
gru_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
gru_model.summary()

In [ ]:
# Training the gru model
gru_model_history = gru_model.fit( x_train, y_train, validation_data = (x_validation,y_validation), epochs = 10, batch_size = 32, callbacks=[early_stop])

# **BIDIRECTIONAL GRU MODEL**

In [ ]:
# Bidirectional GRU model

bidirectional_gru_model = Sequential()
bidirectional_gru_model.add(Embedding(vocab_size,dim,input_length = sent_length))
bidirectional_gru_model.add(Bidirectional(GRU(100)))
bidirectional_gru_model.add(Dense(6,activation = 'softmax'))
bidirectional_gru_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
bidirectional_gru_model.summary()

In [ ]:
# Training the gru model
bidirectional_gru_model_history = bidirectional_gru_model.fit( x_train, y_train, validation_data = (x_validation,y_validation), epochs = 10, batch_size = 32, callbacks=[early_stop])

# **STACKED GRU MODEL**

In [ ]:
# Stacked GRU model

stack_gru_model = Sequential()
stack_gru_model.add(Embedding(vocab_size,dims ,input_length = maxlen))
stack_gru_model.add(GRU(100,return_sequences = True))
stack_gru_model.add(GRU(100,return_sequences = True))
stack_gru_model.add(GRU(50))
stack_gru_model.add(Dense(6,activation = 'softmax'))
stack_gru_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
stack_gru_model.summary()


In [ ]:
# Training the gru model
stack_gru_model_history = stack_gru_model.fit( x_train, y_train, validation_data = (x_validation,y_validation), epochs = 10, batch_size = 32, callbacks=[early_stop])

# **EVALUATION OF ALL SIX MODELS WITH PLOTTING**

In [ ]:
plt.figure(figsize=(8,5))
# Plotting the accuracy plot of Bidirectional LSTM model
plt.subplot(1,2,1)
plt.title("Bidirectional LSTM Model Accuracy")
plt.plot(bidirectional_lstm_model_history.history['accuracy'],label='Accuracy')
plt.plot(bidirectional_lstm_model_history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()


# Plotting the loss plot of Bidirectional LSTM model
plt.subplot(1,2,2)
plt.title("Bidirectional LSTM Model Loss")
plt.plot(bidirectional_lstm_model_history.history['loss'],label='Loss')
plt.plot(bidirectional_lstm_model_history.history['val_loss'],label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Plotting the accuracy plot of LSTM model
plt.figure(figsize=(8,5))
plt.subplot(1,2,1)
plt.title("LSTM Model Accuracy")
plt.plot(lstm_model_history.history['accuracy'],label='Accuracy')
plt.plot(lstm_model_history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Plotting the loss plot of LSTM model
plt.subplot(1,2,2)
plt.title("LSTM Model Loss")
plt.plot(lstm_model_history.history['loss'],label='Loss')
plt.plot(lstm_model_history.history['val_loss'],label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Plotting the accuracy plot of Stack LSTM model
plt.figure(figsize=(8,5))
plt.subplot(1,2,1)
plt.title("Stack LSTM Model Accuracy")
plt.plot(stack_lstm_model_history.history['accuracy'],label='Accuracy')
plt.plot(stack_lstm_model_history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Plotting the loss plot of Stack LSTM model
plt.subplot(1,2,2)
plt.title("Stack LSTM Model Loss")
plt.plot(stack_lstm_model_history.history['loss'],label='Loss')
plt.plot(stack_lstm_model_history.history['val_loss'],label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Plotting the accuracy plot of GRU model
plt.figure(figsize=(8,5))
plt.subplot(1,2,1)
plt.title("GRU Model Accuracy")
plt.plot(gru_model_history.history['accuracy'],label='Accuracy')
plt.plot(gru_model_history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Plotting the loss plot of GRU model
plt.subplot(1,2,2)
plt.title("GRU Model Loss")
plt.plot(gru_model_history.history['loss'],label='Loss')
plt.plot(gru_model_history.history['val_loss'],label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Plotting the accuracy plot of Bidirectional GRU model
plt.figure(figsize=(8,5))
plt.subplot(1,2,1)
plt.title("Bidirectional GRU Model Accuracy")
plt.plot(bidirectional_gru_model_history.history['accuracy'],label='Accuracy')
plt.plot(bidirectional_gru_model_history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Plotting the loss plot of Bidirectional GRU model
plt.subplot(1,2,2)
plt.title("Bidirectional GRU Model Loss")
plt.plot(bidirectional_gru_model_history.history['loss'],label='Loss')
plt.plot(bidirectional_gru_model_history.history['val_loss'],label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

# Plotting the accuracy plot of Stack GRU model
plt.figure(figsize=(8,5))
plt.subplot(1,2,1)
plt.title("Stack GRU Model Accuracy")
plt.plot(stack_gru_model_history.history['accuracy'],label='Accuracy')
plt.plot(stack_gru_model_history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# Plotting the loss plot of Stack GRU model
plt.subplot(1,2,2)
plt.title("Stack GRU Model Loss")
plt.plot(stack_gru_model_history.history['loss'],label='Loss')
plt.plot(stack_gru_model_history.history['val_loss'],label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# **PREDICTIONS OF MODELS**

In [ ]:
# Predicting the output of each model on test data for model evaluation
'''e.g three labels be : 0 = joy, 1 = anger, 2 = sadness
y_pred = [                                                2D numpy arrays outputs=[num_classes, probabilities]
    [0.1, 0.7, 0.2],   # 70% anger → class 1
    [0.8, 0.1, 0.1],   # 80% joy → class 0
    [0.05, 0.05, 0.9]  # 90% sadness → class 2
]
the model gives continuous numbers (probabilities), not labels.We then convert them into discrete class labels (like 0, 1, 2...) so we can check accuracy.
'''

y_pred_lstm = lstm_model.predict(x_test)
y_pred_bilstm = bidirectional_lstm_model.predict(x_test) #Use the trained BiLSTM model to predict the outputs (labels or probabilities) for the unseen test data (x_test)
y_pred_stacklstm = stack_lstm_model.predict(x_test)
y_pred_bigru = bidirectional_gru_model.predict(x_test)
y_pred_gru = gru_model.predict(x_test)
y_pred_stackgru = stack_gru_model.predict(x_test)


In [ ]:
# Converting the continuous output model into discrete classes,Convert probabilities → class indices
'''y_pred= [1 0 2] ->class indices with the highest probability in each row of the input array using np.argmax() into 1D array'''
y_pred_bilstm = np.array([np.argmax(x) for x in y_pred_bilstm])
y_pred_lstm = np.array([np.argmax(x) for x in y_pred_lstm])
y_pred_stacklstm = np.array([np.argmax(x) for x in y_pred_stacklstm])
y_pred_bigru = np.array([np.argmax(x) for x in y_pred_bigru])
y_pred_gru = np.array([np.argmax(x) for x in y_pred_gru])
y_pred_stackgru = np.array([np.argmax(x) for x in y_pred_stackgru])

In [ ]:
#Now we will predict the accuracy score, f1 score, recall, precision score
output= {"Model Name":['Bidirectional LSTM',"LSTM","Stack LSTM", "Bidirectional GRU","GRU","Stack GRU"],
         "Accuracy Score":[accuracy_score(y_test,y_pred_bilstm),accuracy_score(y_test,y_pred_lstm),accuracy_score(y_test,y_pred_stacklstm),accuracy_score(y_test,y_pred_bigru),accuracy_score(y_test,y_pred_gru),accuracy_score(y_test,y_pred_stackgru)],
         "F1 Score(macro)":[f1_score(y_test,y_pred_bilstm,average='macro'),f1_score(y_test,y_pred_lstm,average='macro'),f1_score(y_test,y_pred_stacklstm,average='macro'),f1_score(y_test,y_pred_bigru,average='macro'),f1_score(y_test,y_pred_gru,average='macro'),f1_score(y_test,y_pred_stackgru,average='macro')],
         "Recall Score(macro)":[recall_score(y_test,y_pred_bilstm,average='macro'),recall_score(y_test,y_pred_lstm,average='macro'),recall_score(y_test,y_pred_stacklstm,average='macro'),recall_score(y_test,y_pred_bigru,average='macro'),recall_score(y_test,y_pred_gru,average='macro'),recall_score(y_test,y_pred_stackgru,average='macro')],
         "Precision Score(macro)":[precision_score(y_test,y_pred_bilstm,average='macro'),precision_score(y_test,y_pred_lstm,average='macro'),precision_score(y_test,y_pred_stacklstm,average='macro'),precision_score(y_test,y_pred_bigru,average='macro'),precision_score(y_test,y_pred_gru,average='macro'),precision_score(y_test,y_pred_stackgru,average='macro')],
         "F1 Score(micro)":[f1_score(y_test,y_pred_bilstm,average='micro'),f1_score(y_test,y_pred_lstm,average='micro'),f1_score(y_test,y_pred_stacklstm,average='micro'),f1_score(y_test,y_pred_bigru,average='micro'),f1_score(y_test,y_pred_gru,average='micro'),f1_score(y_test,y_pred_stackgru,average='micro')],
         "Recall Score(micro)":[recall_score(y_test,y_pred_bilstm,average='micro'),recall_score(y_test,y_pred_lstm,average='micro'),recall_score(y_test,y_pred_stacklstm,average='micro'),recall_score(y_test,y_pred_bigru,average='micro'),recall_score(y_test,y_pred_gru,average='micro'),recall_score(y_test,y_pred_stackgru,average='micro')],
         "Precision Score(micro)":[precision_score(y_test,y_pred_bilstm,average='micro'),precision_score(y_test,y_pred_lstm,average='micro'),precision_score(y_test,y_pred_stacklstm,average='micro'),precision_score(y_test,y_pred_bigru,average='micro'),precision_score(y_test,y_pred_gru,average='micro'),precision_score(y_test,y_pred_stackgru,average='micro')],}

output_df = pd.DataFrame(output)

output_df.to_excel("Report of Trained Model.xlsx")

display(output_df)

**Now perform prediction on custom data**

In [ ]:
#making a function that takes input as text and the output it provides gives the 6 emotions
def predict_emotion(text):
  processed_text= preprocess_data(text)
  text_to_sequence= tokenizer.texts_to_sequences([processed_text])[0]
  padded_sequences= pad_sequences([text_to_sequence], maxlen=50, padding='post')
  prediction= bidirectional_gru_model.predict(padded_sequences)
  emotion_labels= ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
  predicted_emotion= emotion_labels[np.argmax(prediction)]
  print('input', text)
  print('Predicted emotion:', predicted_emotion)





In [ ]:
predict_emotion('I am very happy today! yeayy')
predict_emotion('I feel sad today for not going out today')
predict_emotion('I really love myself')
predict_emotion('She hates everyone who demotivates her')
predict_emotion('The movie was so scary , i got terrified')
predict_emotion('Wow! what a pleasent surprise')
predict_emotion('What a beautiful day today')